In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd
import torch

from src import config
from src.console import print_header, print_kv, print_status, print_subheader
from src.model_analysis import load_run_gate_weights, plot_gate_distribution, summarize_gate_values
from src.preprocessing import (extract_segmental_features_cached, extract_suprasegmental_features_cached,
                               mfcc_frame_count)
from src.splits import summarize_severity_loso_splits
from src.training.data import build_speaker_label_map, load_manifest
from src.training.models import MODEL_DESCRIPTIONS, SEVERITY_MODEL_NAME, build_model, parameter_counts
from src.training.reporting import (check_frozen_config_guard, print_feature_audit,
                                    print_final_run_configuration, write_frozen_config)
from src.training.runner import TrainingConfig, run_training
from src.training.utils import resolve_device, set_seed

config.ensure_directories()
df_m6 = load_manifest()

print_header("Three-Branch Gated-Fusion Severity Architecture -- Training")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))

In [ ]:
RUN_NAME = "severity_gated_fusion_three_branch"

cfg = TrainingConfig(
    task="severity",
    model=SEVERITY_MODEL_NAME,
    run_name=RUN_NAME,
    severity_protocol=config.SEVERITY_PRIMARY_PROTOCOL,
)

final_config = print_final_run_configuration(cfg, df_m6)

In [ ]:
print_subheader("Severity dataset -- 15 dysarthric speakers")

speaker_severity = (df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)]
                    .drop_duplicates("Speaker_ID")[["Speaker_ID", "Severity"]])
speaker_counts = speaker_severity["Severity"].value_counts().reindex(config.SEVERITY_CLASS_NAMES)
print_kv("Speakers per severity class", dict(speaker_counts))

utterance_counts = (df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)]
                    .groupby("Severity").size().reindex(config.SEVERITY_CLASS_NAMES))
print_kv("Utterances per severity class", dict(utterance_counts))
speaker_severity.sort_values(["Severity", "Speaker_ID"]).reset_index(drop=True)

In [ ]:
summarize_severity_loso_splits(df_m6)

In [ ]:
device = resolve_device(None)
set_seed(cfg.seed)

# A representative speaker-label map (excludes one held-out speaker) just to
# size the model for the checks below -- the real per-fold map is built
# fresh inside run_training/run_fold for each of the 15 folds.
example_train_df = df_m6[df_m6["Speaker_ID"] != config.DYSARTHRIC_IDS[0]]
example_speaker_map = build_speaker_label_map(example_train_df)

model = build_model(cfg.model, config.NUM_CLASSES[cfg.task],
                    num_speakers=len(example_speaker_map)).to(device)

print_kv("Model", MODEL_DESCRIPTIONS[cfg.model])
print_kv("Device", device)

In [ ]:
counts = parameter_counts(model)
lora_params = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and "lora_" in n)

print_kv("Trainable parameters", f"{counts['trainable_params']:,}")
print_kv("Total parameters", f"{counts['total_params']:,}")
print_kv("Trainable %", f"{counts['trainable_pct']:.2f}%")
print_kv("Frozen parameters", f"{counts['total_params'] - counts['trainable_params']:,}")
print_kv("LoRA adapter parameters", f"{lora_params:,}")

In [ ]:
feature_audit_result = print_feature_audit(model=model, num_classes=config.NUM_CLASSES[cfg.task])

assert feature_audit_result["learned_branch"]["dimensions"] == config.LEARNED_EMBED_DIM == 128
assert feature_audit_result["segmental_branch"]["dimensions"] == config.SEGMENTAL_EMBED_DIM == 64
assert feature_audit_result["suprasegmental_branch"]["dimensions"] == config.SUPRA_EMBED_DIM == 64
assert feature_audit_result["fusion"]["fused_dim"] == config.FUSED_EMBED_DIM == 256
print_status("Branch dimensions match the frozen architecture spec exactly (128 / 64 / 64 -> 256)", ok=True)

In [ ]:
total_frames = mfcc_frame_count(config.MAX_SAMPLES)
dummy_batch = 2

dummy_waveform = torch.zeros(dummy_batch, config.MAX_SAMPLES, device=device)
dummy_mfcc = torch.zeros(dummy_batch, config.SEGMENTAL_CHANNELS, total_frames, device=device)
dummy_supra = torch.zeros(dummy_batch, config.SUPRA_CHANNELS, total_frames, device=device)
dummy_mask = torch.ones(dummy_batch, config.MAX_SAMPLES, dtype=torch.bool, device=device)
dummy_valid_frames = torch.full((dummy_batch,), total_frames, dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    dummy_logits = model(waveform=dummy_waveform, mfcc=dummy_mfcc, attention_mask=dummy_mask,
                         supra=dummy_supra, supra_valid_frames=dummy_valid_frames)

print_kv("Dummy forward output shape", tuple(dummy_logits.shape))
assert dummy_logits.shape == (dummy_batch, config.NUM_CLASSES[cfg.task])
assert torch.isfinite(dummy_logits).all()
print_status("Dummy forward pass produced finite, correctly-shaped logits", ok=True)

In [ ]:
sample_row = df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].iloc[0]

try:
    segmental_features = extract_segmental_features_cached(sample_row["Filepath"])
    supra_features = extract_suprasegmental_features_cached(sample_row["Filepath"])
    print_kv("Sample utterance", sample_row["Filename"])
    print_kv("Segmental feature tensor shape", tuple(segmental_features.shape))
    print_kv("Suprasegmental feature tensor shape", tuple(supra_features.shape))
    assert segmental_features.shape[0] == config.SEGMENTAL_CHANNELS == 43
    assert supra_features.shape[0] == config.SUPRA_CHANNELS == 3
    print_status("Framewise feature extraction produced the expected channel counts", ok=True)
except (FileNotFoundError, RuntimeError, OSError) as exc:
    print_status(f"No audio available in this checkout ({exc}) -- skipping real feature "
                "extraction; the dummy-tensor checks above already validate the architecture.",
                ok=False)

In [ ]:
# COMPUTE BUDGET, STEP 0 -- measured batch-size selection (RTX 4060, 8GB).
# Replaces the comment-documented guess in config.py (DEFAULT_BATCH_SIZE=32,
# "tuned for an 8GB RTX 4060") with a real measurement on THIS machine: one
# real LOSO fold x 1 epoch per candidate batch size, via the same
# run_training path every other check in this notebook uses.
from src.training.budget import benchmark_batch_sizes

batch_bench = benchmark_batch_sizes(df_m6, task="severity", model_name=SEVERITY_MODEL_NAME,
                                    batch_sizes=[16, 24, 32], epochs=1)
batch_bench

In [ ]:
# COMPUTE BUDGET, STEP 1 -- real per-fold-epoch cost, projected against the
# hard wall-clock cap for the one-shot 15-fold severity run
# (config.PRIMARY_SEVERITY_BUDGET_HOURS -- RTX 4060 laptop, 8GB, not a
# second more).
#
# Warms the framewise Praat/MFCC preprocessing cache on a real sample of
# files FIRST, outside the timed benchmark: this project's own development
# confirmed this one-time cost (multiple minutes of single-threaded Praat
# formant/HNR/pitch calls per file, the first time each file is touched) is
# real and non-trivial -- left inside the timed benchmark, it gets baked
# into the "per epoch" measurement and makes the projection pessimistic,
# the same reason ExperimentBudgetManager.benchmark() already warms the
# cacheable-frozen-embedding legacy models' cache before timing them.
#
# epochs_per_fold_estimate is left unset deliberately: ExperimentBudgetManager
# defaults it to config.DEFAULT_PATIENCE + 2, and cfg above (cell 2) was
# built from that same config.py default, so this projection and the
# patience/epoch ceiling run_training actually uses stay consistent by
# construction -- no separate number to keep in sync by hand.
from src.training.budget import ExperimentBudgetManager
from src.preprocessing import extract_segmental_features_cached, extract_suprasegmental_features_cached

_warm_sample = df_m6[df_m6["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].sample(
    n=min(20, len(df_m6)), random_state=config.DEFAULT_SEED)
for _row in _warm_sample.itertuples(index=False):
    extract_segmental_features_cached(_row.Filepath)
    extract_suprasegmental_features_cached(_row.Filepath)

budget = ExperimentBudgetManager(models=[SEVERITY_MODEL_NAME],
                                 hard_cap_hours=config.PRIMARY_SEVERITY_BUDGET_HOURS,
                                 n_folds=15, name="severity_primary")
budget.benchmark(df_m6, task="severity")
budget.preflight()

In [ ]:
# COMPUTE BUDGET, STEP 2 -- confirm the locked epoch/patience budget and
# allocate the session's wall-clock cap across it.
#
# patience=3 (down from the historical default of 5) and epochs=15 (down
# from 20) are no longer set here by mutating cfg -- they are now
# config.DEFAULT_PATIENCE / config.DEFAULT_EPOCHS themselves (see
# src/config.py), so cfg already carries them from the moment it was built
# in cell 2, and STEP 1's budget projection above is already consistent with
# what run_training will actually do. This cell only confirms that and
# performs the allocation.
#
# patience=3 rationale: early stopping on validation loss is this
# architecture's primary anti-overfitting gate -- tightening it stops
# compute being spent past convergence on a 14-speaker per-fold training
# set, which helps "purely learnt, not overfitting" directly, not only
# wall-clock cost. epochs=15 is a ceiling early stopping should trigger well
# before, not a target to reach.
#
# Re-check fold 1's val-loss curve as a canary once the real run starts: if
# it is still clearly decreasing when patience fires, loosen
# final_run_cfg's patience to 4 for the remaining folds (in the MODE gate
# cell below) rather than trust this a priori number blindly.
assert cfg.patience == config.DEFAULT_PATIENCE and cfg.epochs == config.DEFAULT_EPOCHS
budget.allocate()
print_status(f"Compute budget locked: patience={cfg.patience}, epoch ceiling={cfg.epochs}, "
            f"hard cap={config.PRIMARY_SEVERITY_BUDGET_HOURS}h", ok=True)

In [ ]:
SMOKE_RUN_NAME = f"_smoke_test_{RUN_NAME}"
smoke_cfg = TrainingConfig(
    task="severity", model=SEVERITY_MODEL_NAME, run_name=SMOKE_RUN_NAME,
    severity_protocol="full_loso", epochs=1, max_folds=1, limit_samples=16,
    batch_size=4, num_workers=0,
)

print_header("SMOKE TEST -- one fold, one epoch, capped samples. NOT a scientific result.")
smoke_summary, smoke_pooled = run_training(df_m6, smoke_cfg)
print_status("Smoke test completed without error", ok=(not smoke_summary.empty))

In [ ]:
check_frozen_config_guard(final_config)
frozen_path = write_frozen_config(final_config)
print_status(f"Configuration frozen -> {frozen_path}", ok=True)
print_status("The smoke test above proved the pipeline runs end-to-end -- this configuration "
            "is now locked for the one real training run.", ok=True)

In [ ]:
# MODE gate -- mirrors notebooks/legacy/03_detection_ablation_training.ipynb's own safety
# convention. SMOKE (default) never launches the real run. Only a deliberate human edit to
# MODE = "FINAL" plus re-executing this cell starts the one-shot 15-fold experiment --
# see the one-shot training rule (this repository's plan history / project instructions).
MODE = "SMOKE"  # "SMOKE" (safe, default) | "FINAL" (the one real run)

# If STEP 1's canary check (fold 1's val-loss curve) suggested patience=3 is
# cutting folds off before convergence, change ONLY this line before setting
# MODE="FINAL" -- everything downstream (final_run_cfg, the budget
# projection's epochs_per_fold_estimate) reads from it, so it is the single
# place to loosen the anti-overfitting gate if the measurement earns it.
FINAL_PATIENCE = cfg.patience

MODE_SETTINGS = {
    "SMOKE": dict(max_folds=1, limit_samples=16, epochs=1),
    "FINAL": dict(max_folds=None, limit_samples=None, epochs=cfg.epochs),
}
settings = MODE_SETTINGS[MODE]

final_run_cfg = TrainingConfig(
    task="severity", model=SEVERITY_MODEL_NAME, run_name=RUN_NAME,
    severity_protocol="full_loso", epochs=settings["epochs"], patience=FINAL_PATIENCE,
    max_folds=settings["max_folds"], limit_samples=settings["limit_samples"],
)

# The wall-clock compute budget (COMPUTE BUDGET, STEP 0-2 above) is enforced
# HERE, via deadline= -- run_training only stops early when a caller passes
# a deadline (src.training.budget's own docstring); every earlier version of
# this cell omitted it, which meant a FINAL run had no wall-clock ceiling at
# all and could run past config.PRIMARY_SEVERITY_BUDGET_HOURS silently.
# SMOKE mode gets deadline=None (one fold, one epoch -- already cheap enough
# not to need one).
run_deadline = None
if MODE == "FINAL":
    print_header("REAL ONE-SHOT TRAINING RUN")
    run_deadline = budget.deadline_for(SEVERITY_MODEL_NAME, allow_partial=True)
    print_status(f"Compute-budget deadline wired in -- run_training will stop cleanly at a "
                f"fold boundary once the {config.PRIMARY_SEVERITY_BUDGET_HOURS}h session cap "
                "is reached.", ok=True)
else:
    print_status(f"MODE='{MODE}' -- the real 15-fold, {cfg.epochs}-epoch run will NOT execute. "
                "Set MODE='FINAL' and re-run this cell only when ready for the one-shot "
                "experiment.", ok=True)

summary, pooled_metrics = run_training(df_m6, final_run_cfg, deadline=run_deadline)

In [ ]:
metrics_dir = config.METRICS_DIR / RUN_NAME
per_fold_path = metrics_dir / f"{RUN_NAME}.per_fold.csv"

if per_fold_path.exists():
    per_fold_df = pd.read_csv(per_fold_path)
    display_cols = [c for c in ["fold", "epochs_completed", "test_loss", "accuracy",
                                "train_time_s"] if c in per_fold_df.columns]
    per_fold_df[display_cols]
else:
    per_fold_df = pd.DataFrame()
    print_status("No per-fold metrics yet for this run -- run training first.", ok=False)

In [ ]:
pooled_path = config.METRICS_DIR / RUN_NAME / "ALL_FOLDS_pooled.json"

if pooled_path.exists():
    with open(pooled_path) as f:
        pooled = json.load(f)
    for k, v in pooled.items():
        print_kv(k, v)
else:
    print_status("No pooled metrics yet for this run.", ok=False)

In [ ]:
try:
    gate_df = load_run_gate_weights(RUN_NAME)
    gate_summary = summarize_gate_values(gate_df)
    plot_gate_distribution(gate_df, show=True)
    gate_summary
except FileNotFoundError:
    print_status("No gate weights saved yet for this run.", ok=False)

In [ ]:
if not per_fold_df.empty and "complementarity_penalty" in per_fold_df.columns:
    print_kv("Complementarity penalty (mean +/- std)",
             f"{per_fold_df['complementarity_penalty'].mean():.5f} +/- "
             f"{per_fold_df['complementarity_penalty'].std():.5f}")
    print_kv("lambda_comp (frozen)", config.LAMBDA_COMP)
else:
    print_status("No complementarity-penalty diagnostics saved yet for this run.", ok=False)

In [ ]:
if not per_fold_df.empty and "speaker_accuracy" in per_fold_df.columns:
    print_kv("Speaker-head accuracy (mean +/- std)",
             f"{per_fold_df['speaker_accuracy'].mean():.3f} +/- "
             f"{per_fold_df['speaker_accuracy'].std():.3f}")
    print_kv("lambda_speaker (frozen)", config.LAMBDA_SPEAKER)
    print_status("Lower speaker-head accuracy suggests the fused representation carries less "
                "speaker-identifying information -- a diagnostic, not proof of invariance.",
                ok=True)
else:
    print_status("No speaker-adversarial diagnostics saved yet for this run.", ok=False)

In [ ]:
ckpt_dir = config.CHECKPOINT_DIR / RUN_NAME
if ckpt_dir.exists():
    fold_ckpts = sorted(p.parent.name for p in ckpt_dir.glob("*/best.pt"))
    print_kv("Checkpoints saved", f"{len(fold_ckpts)} fold(s)")
    print_kv("Checkpoint directory", ckpt_dir)
    print_kv("Fold IDs", fold_ckpts)
else:
    print_status("No checkpoints saved yet for this run.", ok=False)

print_header("REAL ONE-SHOT TRAINING HAS NOT BEEN EXECUTED (unless MODE was explicitly set to 'FINAL' above)")